# GRASP com Data Mining via K-Means nas soluções elite

### IMPORTS

In [2]:
"""
Fase 1 - GRASP puro (warm-up):
Para cada iteração:
    1. Construição gulosa-aleatória via LRC (Lista Restrita de Candidatos)
    2. Busca Local
    3. Guarda a solução na pool de elites

Fase 2 - DM - GRASP
A cada DM_INTERNAL iterações:
    . Aplica K-Means nas posições das ambulâncias das top-K soluções
    . Os centroides dos custlers viram "locais promissores"
    . A LRC passa a ter bias: locais próximos a centróides têm alpha reduzido
    (i.e., são preferidos mesmo com ganho marginal menor)
    . Novas soluções são construídas com esse bias

Referência base para o DM-GRASP:
    Ribeiro & Souza (2002) - "Variable neighborhood search for the degree-
    constrained minimum spanning tree problem" - padrão de elite set + mining.

"""

import random
import main
import time
from dataclasses import dataclass, field
from typing import List, Tuple, Set

import numpy as np
from sklearn.cluster import KMeans
from baseline.POSSIBLE import DISTRICTS_POINTS as DP

### Parâmetros do problema

In [1]:
NUMBER_AMBUS_TYPE_A = 1
NUMBER_AMBUS_TYPE_B = 4
RAIO = 0.0084
#10min 15min, 30min, 1h
#0.0056 0.0084, 0.0168, 0.0336
BONUS_TYPE_A = 1.20
TOTAL_AMBUS = NUMBER_AMBUS_TYPE_A + NUMBER_AMBUS_TYPE_B

### Parâmetros do DM-GRASP

In [3]:
MAX_ITERATIONS    = 500    # total de iterações GRASP
ALPHA             = 0.20   # grau de aleatoriedade da Lista de Candidatos Restrita (0 = guloso, 1 = aleatório)
ALPHA_BIASED      = 0.05    # alpha mais restrito para locais promissores
LOCAL_SEARCH_ITER = 30      # iterações da busca local por solução
ELITE_SIZE        = 20       # tamanho da pool de elites
DM_INTERVAL       = 50       # a cada N iterações roda K-Means
N_CLUSTERS        = TOTAL_AMBUS  # um cluster por "slot" de ambulância
BIAS_RADIUS       = 0.015    # raio para considerar local "próximo a centroide"
WARM_UP_ITERS     = 50       # iterações iniciais sem mineração


### ESTRUTURA DOS DADOS

In [4]:
NUMBER_LOCATIONS = len(DP)
_keys = list(DP.keys())

COORDS_X = np.array([DP[k][0] for k in _keys])
COORDS_Y = np.array([DP[k][1] for k in _keys])
WEIGHTS = np.array([float(DP[k][2]) if str(DP[k][2]) != "nan" else 0.0 for k in _keys])

COORDS = np.column_stack([COORDS_X, COORDS_Y])
TOTAL_DEMAND = float(WEIGHTS.sum())


@dataclass
class Solution:
    positions_A: List[int] = field(default_factory=list)   # índices TypeA
    positions_B: List[int] = field(default_factory=list)   # índices TypeB
    fitness: float = 0.0
    coverage: float = 0.0   # % real de demanda coberta

    def all_positions(self) -> List[int]:
        return self.positions_A + self.positions_B

    def __lt__(self, other): return self.fitness < other.fitness
    def __gt__(self, other): return self.fitness > other.fitness

    

### Pré-computação da matriz de distâncias

In [5]:
def build_distance_matrix() -> np.array:
    diff_x = COORDS_X[: None] - COORDS_X[None, :] 
    diff_y = COORDS_Y[: None] - COORDS_Y[None, :]
    return np.sqrt(diff_x**2, diff_y**2)